In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os
from alpaca.data import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame, TimeFrameUnit
import pandas as pd
import logging
import exchange_calendars as ec

In [9]:
logging.basicConfig(
    filename='stocks_dl_volume.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='a'
)
logger = logging.getLogger(__name__)

In [3]:
load_dotenv()

True

In [4]:
# Load secrets from environment variable
ALPACA_KEY = os.getenv("ALPACA_KEY")
ALPACA_SECRET = os.getenv("ALPACA_SECRET")
client = StockHistoricalDataClient(ALPACA_KEY, ALPACA_SECRET)

Configure directory to store data.  This repo doesn't come with data, but has code showing how to download it.

In [5]:
PROJECT_DATA = Path(os.getenv('PROJECT_DATA'))
PROJECT_DATA.mkdir(parents=True, exist_ok=True)

Alpaca has data outside of regular trading hours.  We want to fit the data to an exchange calendar
that trades from 9:30 am to 4:00 pm EST.  Data outside of the regular window gets dropped.

In [6]:
xnys = ec.get_calendar("XNYS")
sched = xnys.schedule.loc['2026-01-01':'2026-03-31']
trading_index = pd.DatetimeIndex([], tz='UTC')

for index, row in sched.iterrows():
    index_segment = pd.date_range(row['open'], row['close'], freq="1min", tz='UTC')
    trading_index = trading_index.union(index_segment) 

trading_index

DatetimeIndex(['2026-01-02 14:30:00+00:00', '2026-01-02 14:31:00+00:00',
               '2026-01-02 14:32:00+00:00', '2026-01-02 14:33:00+00:00',
               '2026-01-02 14:34:00+00:00', '2026-01-02 14:35:00+00:00',
               '2026-01-02 14:36:00+00:00', '2026-01-02 14:37:00+00:00',
               '2026-01-02 14:38:00+00:00', '2026-01-02 14:39:00+00:00',
               ...
               '2026-03-31 19:51:00+00:00', '2026-03-31 19:52:00+00:00',
               '2026-03-31 19:53:00+00:00', '2026-03-31 19:54:00+00:00',
               '2026-03-31 19:55:00+00:00', '2026-03-31 19:56:00+00:00',
               '2026-03-31 19:57:00+00:00', '2026-03-31 19:58:00+00:00',
               '2026-03-31 19:59:00+00:00', '2026-03-31 20:00:00+00:00'],
              dtype='datetime64[ns, UTC]', length=23851, freq=None)

In [10]:
def clean(index, df):
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"Passed DF doesn't have Datetime index")

    df_ridx = df.reindex(index)
    num_na_ridx = df_ridx.isnull().sum().sum()
    num_exist_ridx = df_ridx.count().sum()
    na_ridx_pct = num_na_ridx / (num_na_ridx + num_exist_ridx)
    if na_ridx_pct > .05:
        logger.error(f"{ticker} DF contains > 5% nans.")
        return None
    else:
        df_cleaned = df_ridx.ffill().dropna()
        return df_cleaned

ticker = 'AAPL'
file_path = PROJECT_DATA / f'{ticker}.feather'
logger.info(f'Requesting data...')
request_params_min = StockBarsRequest(
                    symbol_or_symbols=ticker,
                    timeframe=TimeFrame(1, TimeFrameUnit.Minute),
                    start='2026-01-01',
                    end='2026-04-01 00:00', # up to and excluding
                    adjustment='split',
                    sort='asc'
            )
try:    
    bars_response = client.get_stock_bars(request_params_min)
except:
    logger.error(f"Could not request ticker {ticker}")
df = bars_response.df
# Most dataframes have a multi level index.  We don't need level 0.
try:
    df.index = df.index.droplevel(0)
except:
    logger.error(f"Could not drop level for ticker {ticker}")
    with open('no_drop.txt', 'a') as file:
        file.write(ticker + '\n')
df = df.drop(columns=['trade_count', 'vwap'])
df_cleaned = clean(trading_index, df)
if df_cleaned is None:
    logger.warning(f'Ticker {ticker} was rejected for too many nans')
else:
    df_cleaned.to_feather(file_path)
    logger.info(f"Wrote {ticker} to feather")